# Qwen 3.5 4B ile Türkçe GSM8K Matematik Problemleri Değerlendirmesi

Bu notebook, `Qwen/Qwen3.5-4B` modelinin Türkçe matematik kelime problemlerini (GSM8K) çözme yeteneğini değerlendirmek için hazırlanmıştır.

Modelin ürettiği metin içerisinden nihai sayısal cevabı Regex kullanarak çıkaran özel bir ayrıştırıcı (parser) içerir ve modelin doğruluğunu otomatik olarak test eder. Çalışma sonucunda başarılı ve başarısız olan çıkarımlar analiz edilmek üzere CSV formatında dışa aktarılır.

## 1. Kurulum ve Gereksinimler
Modeli ve veri setini yüklemek için gerekli kütüphaneleri kuruyoruz. Qwen modellerinin düzgün çalışması için `transformers` kütüphanesinin güncel olması önemlidir.

In [ ]:
!pip install -q -U "transformers>=4.51.0" datasets accelerate pandas

## 2. Kütüphanelerin İçe Aktarılması

In [ ]:
import torch
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

## 3. Veri Setinin Yüklenmesi
Hugging Face üzerinde bulunan ve orijinal GSM8K veri setinin Türkçeye çevrilmiş hali olan `ytu-ce-cosmos/gsm8k_tr` veri setinin eğitim (train) bölümünü kullanıyoruz.

In [ ]:
print('Veri seti yükleniyor...')
dataset = load_dataset('ytu-ce-cosmos/gsm8k_tr', split='train')

## 4. Model ve Tokenizer Yükleme
Bu projede `Qwen/Qwen3.5-4B` modeli kullanılmaktadır. Bellek kullanımını optimize etmek amacıyla model `float16` hassasiyetinde yüklenmiştir.

In [ ]:
model_id = 'Qwen/Qwen3.5-4B'

print('Tokenizer yükleniyor...')
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left'

print('Model yükleniyor (birkaç dakika sürebilir)...')
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto'
)
print('Model hazır!')

## 5. Yardımcı Fonksiyonlar

Modelin ürettiği metin tabanlı yanıtlardan matematiksel sonucu otomatize bir şekilde ayıklamak için iki temel fonksiyona ihtiyacımız var:

1. **`extract_final_number`**: Modelin "Sonuç: X" formatında verdiği yanıtları yakalar. Türkçe binlik ayraçları ve ondalık dilimlemeleri temizleyerek float/int formatına dönüştürür.
2. **`get_model_answer`**: Verilen soruyu modelin beklediği Chat formatına (Jinja template) sokar ve çıkarım (inference) yapar.

In [ ]:
def extract_final_number(text):
    # 1. "Sonuç: 216" veya "Cevap: 216" kalıbına öncelik ver
    match = re.search(r'[Ss]onu[cç]\s*[:\=]\s*(-?\d+(?:[.,]\d+)*)', str(text))

    # Eğer "Sonuç:" kalıbı yoksa (model unuttuysa), metindeki en son sayıyı bul
    if not match:
        numbers = re.findall(r'-?\d+(?:[.,]\d+)*', str(text))
        if numbers:
            raw_num = numbers[-1]
        else:
            return None
    else:
        raw_num = match.group(1)

    # 2. Sayıyı Temizleme (Türkçe Binlik Ayraç vs Ondalık)
    # Eğer sayı "11.000" veya "1,000" gibi 3'lü gruplanmış bir binlik ayraç içeriyorsa, ayraçları tamamen sil
    if re.match(r'^-?\d{1,3}([.,]\d{3})+$', raw_num):
        raw_num = raw_num.replace('.', '').replace(',', '')

    # Geriye kalan (varsa) virgülleri ondalık noktaya çevir (Örn: 2,50 -> 2.50)
    raw_num = raw_num.replace(',', '.')

    # 3. Float/Integer Dönüşümü
    try:
        val = float(raw_num)
        return int(val) if val.is_integer() else val
    except ValueError:
        return None

def get_model_answer(question):
    messages = [
        {'role': 'system', 'content': 'Sen bir matematik problemi çözen asistansın. Problemi adım adım çöz ve cevabını "Sonuç: <sayı>" formatında ver.'},
        {'role': 'user', 'content': question}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

## 6. Hızlı Test (Sanity Check)
Değerlendirme döngüsüne girmeden önce, sistemin doğru çalıştığından emin olmak için veri setindeki ilk 3 soruyu test ediyoruz.

In [ ]:
print('--- ILK 3 SORU TESTI ---\n')
for i, item in enumerate(dataset.select(range(3))):
    question = item['question']
    expected_number = extract_final_number(item['answer'])
    response = get_model_answer(question)
    model_number = extract_final_number(response)

    print(f'[{i+1}] SORU: {question}')
    print(f'    BEKLENEN : {expected_number}')
    print(f'    MODEL CEVABI: {response}')
    print(f'    MODELDEN ALINAN: {model_number}')
    sonuc = '✅ BASARILI' if model_number == expected_number else '❌ BASARISIZ'
    print(f'    SONUC: {sonuc}')
    print('=' * 60)

## 7. Ana Değerlendirme Döngüsü
Modelin tutarlılığını analiz edebilmek için veri seti üzerinde toplu (batch) değerlendirme yapıyoruz. Hedefimiz analiz için **50 başarılı** ve **50 başarısız** örnek toplamaktır. Bu hedeflere ulaşıldığında döngü erken sonlandırılır (early exit).

In [ ]:
successful_samples = []
unsuccessful_samples = []

BATCH_SIZE = 16
TARGET = 50

print('Sorular test ediliyor...')

all_items = list(dataset)
batch_num = 0

for start in range(0, len(all_items), BATCH_SIZE):
    # Hedef dolmuşsa erken çıkış yap
    if len(successful_samples) >= TARGET and len(unsuccessful_samples) >= TARGET:
        break

    batch_items = all_items[start:start + BATCH_SIZE]
    batch_num += 1

    # Promptları hazırla
    texts = []
    for item in batch_items:
        messages = [
            {'role': 'system', 'content': 'Sen bir matematik problemi çözen asistansın. Problemi adım adım çöz ve cevabını "Sonuç: <sayı>" formatında ver.'},
            {'role': 'user', 'content': item['question']}
        ]
        texts.append(tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=True, enable_thinking=False
        ))

    # Toplu olarak modele gönder (Batch Inference)
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    # Sonuçları değerlendir ve kategorize et
    for item, out in zip(batch_items, outputs):
        expected_number = extract_final_number(item['answer'])
        if expected_number is None:
            continue

        response = tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        model_number = extract_final_number(response)

        sample_data = {
            'question': item['question'],
            'expected_answer_text': item['answer'],
            'expected_number': expected_number,
            'model_response': response,
            'extracted_model_number': model_number
        }

        if model_number == expected_number:
            if len(successful_samples) < TARGET:
                successful_samples.append(sample_data)
                print(f'✅ Başarılı: {len(successful_samples)}/{TARGET}')
        else:
            if len(unsuccessful_samples) < TARGET:
                unsuccessful_samples.append(sample_data)
                print(f'❌ Başarısız: {len(unsuccessful_samples)}/{TARGET}')

    print(f'  Batch {batch_num} tamamlandı ({min(start+BATCH_SIZE, len(all_items))}/{len(all_items)}) | ✅ {len(successful_samples)} ❌ {len(unsuccessful_samples)}')

print(f'\nToplam başarılı: {len(successful_samples)}')
print(f'Toplam başarısız: {len(unsuccessful_samples)}')

## 8. Sonuçların Dışa Aktarılması
Toplanan başarılı ve başarısız analiz verileri, detaylı inceleme (error analysis) yapılabilmesi için ayrık CSV dosyaları olarak kaydedilir.

In [ ]:
pd.DataFrame(successful_samples).to_csv('qwen3_5_4b_basarili_50.csv', index=False)
pd.DataFrame(unsuccessful_samples).to_csv('qwen3_5_4b_basarisiz_50.csv', index=False)
print('Tamamlandı! CSV dosyaları kaydedildi.')